# OpenAI Embeddings + Cosine Similarity

In [5]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import numpy as np

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Embed a single piece of text

In [6]:
vector1 = embeddings.embed_query("The capital of India is Delhi")
vector2 = embeddings.embed_query("phosynthesis is a science")
vector3 = embeddings.embed_query(" newton provide 3 laws")
vector4 = embeddings.embed_query("aryabhtta invent zero")
vector5 = embeddings.embed_query("vedant is the freind of sadiq")


In [3]:
query= "who is the freind of sadiq"

In [7]:
r45=cosine_similarity(vector4,vector5)
r45

np.float64(0.23589844807598373)

In [8]:
len(vector4)

3072

In [2]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [13]:
r12=cosine_similarity(vector1,vector2)
float(r12)

0.048307325913571966

In [10]:
r23=cosine_similarity(vector2,vector3)
r23

np.float64(0.1693856249731608)

## 2. Embed multiple documents

In [14]:
texts = [
    "The cat sat on the mat.",
    "A dog played in the park.",
    "The sun is shining brightly.",
    "I love to read books."
    
]

doc_vectors = embeddings.embed_documents(texts)

print(len(doc_vectors), len(doc_vectors[1]))

4 3072


## 3. Cosine similarity from scratch

In [15]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

## 4. Compare a query against the documents

In [21]:
query = "A kitten is resting on a rug."
query_vector = embeddings.embed_query(query)

for text, doc_vector in zip(texts, doc_vectors):
    score = cosine_similarity(query_vector, doc_vector)
    print(f"{(score):.4f}  {text}")

0.4251  The cat sat on the mat.
0.1996  A dog played in the park.
0.1394  The sun is shining brightly.
0.0972  I love to read books.


## 5. Find the most similar document

In [27]:
scores = [cosine_similarity(query_vector, v) for v in doc_vectors]
best_idx = np.int64(np.argmax(scores))

print(f"Most similar: {texts[best_idx]!r} (score={scores[best_idx]:.4f})")

Most similar: 'The cat sat on the mat.' (score=0.4251)


## 6. Similarity matrix across all documents

In [28]:
matrix = np.array(
    [[cosine_similarity(a, b) for b in doc_vectors] for a in doc_vectors]
)

print(np.round(matrix, 2))

[[1.   0.26 0.16 0.17]
 [0.26 1.   0.2  0.1 ]
 [0.16 0.2  1.   0.11]
 [0.17 0.1  0.11 1.  ]]


## 7. Top-k retrieval for multiple queries

In [32]:
queries = [
    "How is the economy doing?",
    "Tell me about pets.",
]

query_vectors = embeddings.embed_documents(queries)

for q, qv in zip(queries, query_vectors):
    scores = [cosine_similarity(qv, dv) for dv in doc_vectors]
    top_idx = int(np.argmax(scores))
    print(f"{q!r} - {texts[top_idx]!r} (score={scores[top_idx]:.4f})")

'How is the economy doing?' - 'The sun is shining brightly.' (score=0.1594)
'Tell me about pets.' - 'The cat sat on the mat.' (score=0.2519)


## 8. Near-duplicate detection with a threshold

In [33]:
pair_a = "The quick brown fox jumps over the lazy dog."
pair_b = "A fast brown fox leaps over a sleepy dog."

vec_a, vec_b = embeddings.embed_documents([pair_a, pair_b])
score = cosine_similarity(vec_a, vec_b)

threshold = 0.9
print(f"score={score:.4f}")
print("near-duplicate" if score >= threshold else "not a duplicate")

score=0.7736
not a duplicate
